In [1]:
import io
import zipfile
from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional, Tuple

import requests

# =========================
# CONFIG
# =========================
TOKENS_ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")
MAX_TOKENS_TO_USE = 7

CONNECT_TIMEOUT_S = 10
READ_TIMEOUT_S = 120

RUN_ID = 20620229392
FULL_NAME = "Bdaya-Dev/oidc"


# Save here (your requested folder)
OUT_BASE = Path(r"C:\Android Mobile App\ICST2026_Ext\log")
OUT_BASE.mkdir(parents=True, exist_ok=True)

OUT_ZIP = OUT_BASE / f"run_{RUN_ID}_logs.zip"
OUT_DIR = OUT_BASE / f"run_{RUN_ID}_logs_extracted"

# =========================
# Token loader (same as Stage code)
# =========================
def load_tokens_from_env_file(env_path: Path, max_tokens: int = 3) -> List[str]:
    if not env_path.exists():
        raise FileNotFoundError(f"Tokens env file not found: {env_path}")
    tokens: List[str] = []
    for line in env_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        k = k.strip()
        v = v.strip().strip('"').strip("'")
        if k.startswith("GITHUB_TOKEN_") and v:
            tokens.append(v)
            if len(tokens) >= max_tokens:
                break
    if not tokens:
        raise ValueError(f"No tokens found in {env_path}. Expected keys like GITHUB_TOKEN_1=...")
    return tokens

# =========================
# Minimal GitHub client
# =========================
@dataclass
class TokenState:
    token: str
    remaining: Optional[int] = None

class GitHubClient:
    def __init__(self, tokens: List[str]) -> None:
        self.session = requests.Session()
        self.session.headers.update({
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "log-downloader/1.0",
        })
        self.tokens = [TokenState(t) for t in tokens]

    def _pick_idx(self) -> int:
        for i, st in enumerate(self.tokens):
            if st.remaining is None or st.remaining > 0:
                return i
        return 0

    def request_logs_zip(self, full_name: str, run_id: int) -> Tuple[Optional[bytes], str]:
        url = f"https://api.github.com/repos/{full_name}/actions/runs/{run_id}/logs"

        idx = self._pick_idx()
        st = self.tokens[idx]
        self.session.headers["Authorization"] = f"Bearer {st.token}"

        try:
            r = self.session.get(url, timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S), allow_redirects=False)
        except requests.exceptions.RequestException:
            return None, "error_request"

        rem = r.headers.get("X-RateLimit-Remaining")
        if rem is not None:
            try:
                st.remaining = int(rem)
            except Exception:
                pass

        if r.status_code == 404:
            return None, "not_found"
        if r.status_code == 403:
            return None, "forbidden"

        # Usually 302 redirect to signed URL
        if r.status_code in (301, 302, 303, 307, 308):
            loc = r.headers.get("Location") or r.headers.get("location")
            if not loc:
                return None, f"error_{r.status_code}_no_location"
            try:
                r2 = requests.get(loc, timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S))
            except requests.exceptions.RequestException:
                return None, "error_signed_request"
            if r2.status_code == 200:
                return r2.content, "ok"
            if r2.status_code == 404:
                return None, "not_found"
            if r2.status_code == 403:
                return None, "forbidden"
            return None, f"error_signed_{r2.status_code}"

        if r.status_code == 200:
            return r.content, "ok"

        return None, f"error_{r.status_code}"

# =========================
# Main
# =========================
def main() -> None:
    tokens = load_tokens_from_env_file(TOKENS_ENV_PATH, max_tokens=MAX_TOKENS_TO_USE)
    gh = GitHubClient(tokens)

    data, status = gh.request_logs_zip(FULL_NAME, RUN_ID)
    print("download status:", status)
    if status != "ok" or not data:
        raise SystemExit("Failed to download logs zip.")

    OUT_ZIP.write_bytes(data)
    print("saved zip:", OUT_ZIP)

    if OUT_DIR.exists():
        # If you rerun, keep things clean
        for p in OUT_DIR.rglob("*"):
            if p.is_file():
                p.unlink()
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(io.BytesIO(data), "r") as z:
        z.extractall(OUT_DIR)

    print("extracted to:", OUT_DIR)

if __name__ == "__main__":
    main()


download status: ok
saved zip: C:\Android Mobile App\ICST2026_Ext\log\run_20620229392_logs.zip
extracted to: C:\Android Mobile App\ICST2026_Ext\log\run_20620229392_logs_extracted


In [2]:
## GitHub API check

In [4]:
import requests
import pandas as pd
from datetime import datetime, timezone

# ============================================================
# CONFIG — this sample
# ============================================================
OWNER = "CatimaLoyalty"
REPO = "Android"
RUN_ID = 20408974923

# Expected study row context for manual review
EXPECTED_STYLE = "Community"   # adjust if your sample row says otherwise
EXPECTED_ATTEMPT = 1           # adjust from your sample sheet if needed

# Optional: add token if you hit rate limits
GITHUB_TOKEN = None  # e.g., "ghp_xxx"

# Current GitHub REST API docs recommend these headers
API_VERSION = "2026-03-10"
BASE = "https://api.github.com"

# ============================================================
# Helpers
# ============================================================
def gh_get(url, params=None):
    headers = {
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": API_VERSION,
    }
    if GITHUB_TOKEN:
        headers["Authorization"] = f"Bearer {GITHUB_TOKEN}"

    r = requests.get(url, headers=headers, params=params, timeout=60)
    r.raise_for_status()
    return r.json()

def parse_dt(x):
    if not x:
        return None
    return datetime.fromisoformat(x.replace("Z", "+00:00"))

def seconds_between(a, b):
    da = parse_dt(a)
    db = parse_dt(b)
    if da is None or db is None:
        return None
    return (db - da).total_seconds()

def looks_like_android_integration(job_name, steps):
    text_parts = [job_name or ""]
    for st in steps or []:
        text_parts.append(st.get("name", "") or "")
    text = " | ".join(text_parts).lower()

    signals = [
        "android-emulator-runner",
        "android emulator",
        "integration tests",
        "flutter test integration_test",
        "avd",
        "emulator",
        "connectedandroidtest",
        "connectedcheck",
        "am instrument",
    ]
    return any(sig in text for sig in signals)

def detect_style(job_name, steps):
    text_parts = [job_name or ""]
    for st in steps or []:
        text_parts.append(st.get("name", "") or "")
        if st.get("name"):
            pass
    text = " | ".join(text_parts).lower()

    if "browserstack" in text or "sauce" in text or "kobiton" in text or "firebase test lab" in text:
        return "Third-Party"
    if "managed device" in text or "gmd" in text:
        return "GMD"
    if "android-emulator-runner" in text or "avd" in text or "emulator" in text:
        return "Community"
    if "detox" in text or "integration test" in text or "flutter test integration_test" in text:
        # This could still appear inside Community if emulator-runner is used,
        # so Community should already have matched above.
        return "Custom"
    return "Unknown"

# ============================================================
# Fetch run metadata
# ============================================================
run_url = f"{BASE}/repos/{OWNER}/{REPO}/actions/runs/{RUN_ID}"
run_data = gh_get(run_url)

# ============================================================
# Fetch jobs for run
# ============================================================
jobs_url = f"{BASE}/repos/{OWNER}/{REPO}/actions/runs/{RUN_ID}/jobs"
jobs_data = gh_get(jobs_url, params={"per_page": 100})

jobs = jobs_data.get("jobs", [])
if not jobs:
    raise RuntimeError("No jobs returned for this run.")

jobs_df = pd.DataFrame([
    {
        "job_id": j.get("id"),
        "name": j.get("name"),
        "status": j.get("status"),
        "conclusion": j.get("conclusion"),
        "started_at": j.get("started_at"),
        "completed_at": j.get("completed_at"),
        "html_url": j.get("html_url"),
        "duration_seconds": seconds_between(j.get("started_at"), j.get("completed_at")),
    }
    for j in jobs
]).sort_values("started_at", na_position="last")

print("\n=== RUN METADATA ===")
print({
    "run_id": run_data.get("id"),
    "run_number": run_data.get("run_number"),
    "run_attempt": run_data.get("run_attempt"),
    "status": run_data.get("status"),
    "conclusion": run_data.get("conclusion"),
    "event": run_data.get("event"),
    "html_url": run_data.get("html_url"),
})

print("\n=== JOBS IN RUN ===")
print(jobs_df.to_string(index=False))

# ============================================================
# Pick instrumentation-related job
# For this sample we expect the 'android' job
# ============================================================
android_job = None

# prefer exact job name 'android'
for j in jobs:
    if (j.get("name") or "").strip().lower() == "android":
        android_job = j
        break

# fallback: look for Android integration signals
if android_job is None:
    for j in jobs:
        if looks_like_android_integration(j.get("name", ""), j.get("steps", [])):
            android_job = j
            break

if android_job is None:
    raise RuntimeError("Could not identify instrumentation-related Android job.")

android_steps_df = pd.DataFrame([
    {
        "number": st.get("number"),
        "name": st.get("name"),
        "status": st.get("status"),
        "conclusion": st.get("conclusion"),
        "started_at": st.get("started_at"),
        "completed_at": st.get("completed_at"),
        "duration_seconds": seconds_between(st.get("started_at"), st.get("completed_at")),
    }
    for st in (android_job.get("steps") or [])
])

print("\n=== INSTRUMENTATION-RELATED JOB ===")
print({
    "job_id": android_job.get("id"),
    "job_name": android_job.get("name"),
    "started_at": android_job.get("started_at"),
    "completed_at": android_job.get("completed_at"),
    "duration_seconds": seconds_between(android_job.get("started_at"), android_job.get("completed_at")),
    "html_url": android_job.get("html_url"),
})

print("\n=== STEPS IN INSTRUMENTATION-RELATED JOB ===")
print(android_steps_df.to_string(index=False))

# ============================================================
# Manual-review oriented interpretation
# ============================================================
observed_style = detect_style(android_job.get("name", ""), android_job.get("steps", []))

# Study-facing in-scope logic for this sample:
# - Android-related instrumentation/integration workflow
# - Not iOS-only, web-only, etc.
in_scope = "Yes" if observed_style in {"Community", "Custom", "GMD", "Third-Party"} else "No"

# Instrumentation executed:
# Here we use the presence of the Android job + steps that clearly run Android integration testing
step_names = " | ".join((st.get("name") or "") for st in (android_job.get("steps") or [])).lower()
instr_executed = "Yes" if (
    "run android emulator and integration tests" in step_names
    or "flutter test integration_test" in step_names
    or looks_like_android_integration(android_job.get("name", ""), android_job.get("steps", []))
) else "No"

style_correct = "Yes" if observed_style == EXPECTED_STYLE else "No"

run_attempt = run_data.get("run_attempt")
if EXPECTED_ATTEMPT == 1:
    # GitHub usually does not display attempt=1 explicitly in the UI,
    # but API does give run_attempt. For API-based check:
    attempt_correct = "Yes" if run_attempt == 1 else "No"
else:
    attempt_correct = "Yes" if run_attempt == EXPECTED_ATTEMPT else "No"

print("\n=== MANUAL REVIEW SUMMARY FOR THIS RECORD ===")
print({
    "audit_in_scope": in_scope,
    "audit_instrumentation_executed": instr_executed,
    "audit_style_correct": style_correct,
    "audit_attempt_correct": attempt_correct,
    "layer1_instrumentation_envelope_start": android_job.get("started_at"),
    "layer1_instrumentation_envelope_end": android_job.get("completed_at"),
    "layer1_instrumentation_envelope_duration_seconds": seconds_between(
        android_job.get("started_at"),
        android_job.get("completed_at")
    ),
})

# Optional: save outputs
jobs_df.to_csv("sample_run_jobs.csv", index=False)
android_steps_df.to_csv("sample_android_job_steps.csv", index=False)

summary_df = pd.DataFrame([{
    "owner": OWNER,
    "repo": REPO,
    "run_id": RUN_ID,
    "run_url": run_data.get("html_url"),
    "run_number": run_data.get("run_number"),
    "run_attempt_api": run_data.get("run_attempt"),
    "run_status": run_data.get("status"),
    "run_conclusion": run_data.get("conclusion"),
    "event": run_data.get("event"),
    "instrumentation_job_id": android_job.get("id"),
    "instrumentation_job_name": android_job.get("name"),
    "instrumentation_job_started_at": android_job.get("started_at"),
    "instrumentation_job_completed_at": android_job.get("completed_at"),
    "instrumentation_job_duration_seconds": seconds_between(
        android_job.get("started_at"),
        android_job.get("completed_at")
    ),
    "observed_style_from_job": observed_style,
    "audit_in_scope": in_scope,
    "audit_instrumentation_executed": instr_executed,
    "audit_style_correct": style_correct,
    "audit_attempt_correct": attempt_correct,
}])
summary_df.to_csv("sample_manual_review_summary.csv", index=False)

print("\nSaved:")
print(" - sample_run_jobs.csv")
print(" - sample_android_job_steps.csv")
print(" - sample_manual_review_summary.csv")


=== RUN METADATA ===
{'run_id': 20408974923, 'run_number': 5859, 'run_attempt': 1, 'status': 'completed', 'conclusion': 'success', 'event': 'push', 'html_url': 'https://github.com/CatimaLoyalty/Android/actions/runs/20408974923'}

=== JOBS IN RUN ===
     job_id          name    status conclusion           started_at         completed_at                                                                          html_url  duration_seconds
58642878071 build (Gplay) completed    success 2025-12-21T11:12:56Z 2025-12-21T11:24:01Z https://github.com/CatimaLoyalty/Android/actions/runs/20408974923/job/58642878071             665.0
58642878075  build (Foss) completed    success 2025-12-21T11:12:56Z 2025-12-21T11:24:11Z https://github.com/CatimaLoyalty/Android/actions/runs/20408974923/job/58642878075             675.0


RuntimeError: Could not identify instrumentation-related Android job.

In [5]:
# check the Job details

In [6]:
import requests

url = "https://api.github.com/repos/CatimaLoyalty/Android/actions/runs/20408974923/jobs"
headers = {"Accept": "application/vnd.github+json"}

r = requests.get(url, headers=headers, timeout=60)
r.raise_for_status()
data = r.json()

for job in data.get("jobs", []):
    print(
        job.get("id"),
        job.get("name"),
        job.get("started_at"),
        job.get("completed_at"),
        job.get("html_url"),
    )

58642878071 build (Gplay) 2025-12-21T11:12:56Z 2025-12-21T11:24:01Z https://github.com/CatimaLoyalty/Android/actions/runs/20408974923/job/58642878071
58642878075 build (Foss) 2025-12-21T11:12:56Z 2025-12-21T11:24:11Z https://github.com/CatimaLoyalty/Android/actions/runs/20408974923/job/58642878075


In [7]:
import requests
import pandas as pd
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================
OWNER = "bluefireteam"
REPO = "audioplayers"
RUN_ID = 20469290792

# Put your token here if needed for rate limits/private repos
GITHUB_TOKEN = None  # e.g. "ghp_xxx"

OUT_DIR = Path("audit_run_20469290792_timeline")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# API helpers
# ============================================================
def gh_headers():
    headers = {
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
    }
    if GITHUB_TOKEN:
        headers["Authorization"] = f"Bearer {GITHUB_TOKEN}"
    return headers

def gh_get(url, params=None):
    r = requests.get(url, headers=gh_headers(), params=params, timeout=60)
    r.raise_for_status()
    return r.json()

def get_run(owner, repo, run_id):
    url = f"https://api.github.com/repos/{owner}/{repo}/actions/runs/{run_id}"
    return gh_get(url)

def list_jobs(owner, repo, run_id):
    jobs = []
    page = 1
    while True:
        url = f"https://api.github.com/repos/{owner}/{repo}/actions/runs/{run_id}/jobs"
        js = gh_get(url, params={"per_page": 100, "page": page})
        arr = js.get("jobs", [])
        if not arr:
            break
        jobs.extend(arr)
        if len(arr) < 100:
            break
        page += 1
    return jobs

# ============================================================
# Main extraction
# ============================================================
run_js = get_run(OWNER, REPO, RUN_ID)
jobs = list_jobs(OWNER, REPO, RUN_ID)

# Save raw run metadata
pd.DataFrame([{
    "run_id": run_js.get("id"),
    "run_number": run_js.get("run_number"),
    "run_attempt": run_js.get("run_attempt"),
    "status": run_js.get("status"),
    "conclusion": run_js.get("conclusion"),
    "event": run_js.get("event"),
    "workflow_id": run_js.get("workflow_id"),
    "run_started_at": run_js.get("run_started_at"),
    "updated_at": run_js.get("updated_at"),
    "head_sha": run_js.get("head_sha"),
    "html_url": run_js.get("html_url"),
}]).to_csv(OUT_DIR / "run_metadata.csv", index=False)

# ------------------------------------------------------------
# Jobs table
# ------------------------------------------------------------
job_rows = []
step_rows = []

for job in jobs:
    job_rows.append({
        "job_id": job.get("id"),
        "job_name": job.get("name"),
        "job_status": job.get("status"),
        "job_conclusion": job.get("conclusion"),
        "job_started_at": job.get("started_at"),
        "job_completed_at": job.get("completed_at"),
        "job_html_url": job.get("html_url"),
        "runner_name": job.get("runner_name"),
        "labels": ",".join(job.get("labels", [])) if isinstance(job.get("labels"), list) else "",
    })

    for st in job.get("steps", []):
        step_rows.append({
            "job_id": job.get("id"),
            "job_name": job.get("name"),
            "job_started_at": job.get("started_at"),
            "job_completed_at": job.get("completed_at"),
            "job_html_url": job.get("html_url"),
            "step_number": st.get("number"),
            "step_name": st.get("name"),
            "step_status": st.get("status"),
            "step_conclusion": st.get("conclusion"),
            "step_started_at": st.get("started_at"),
            "step_completed_at": st.get("completed_at"),
        })

jobs_df = pd.DataFrame(job_rows)
steps_df = pd.DataFrame(step_rows)

# Save jobs
jobs_df = jobs_df.sort_values(["job_started_at", "job_name"], na_position="last").reset_index(drop=True)
jobs_df.to_csv(OUT_DIR / "run_jobs.csv", index=False)

# Save all steps in actual time order
steps_df["step_started_at_dt"] = pd.to_datetime(steps_df["step_started_at"], errors="coerce", utc=True)
steps_df["step_completed_at_dt"] = pd.to_datetime(steps_df["step_completed_at"], errors="coerce", utc=True)
steps_df["duration_seconds"] = (
    steps_df["step_completed_at_dt"] - steps_df["step_started_at_dt"]
).dt.total_seconds()

steps_df = steps_df.sort_values(
    ["step_started_at_dt", "step_completed_at_dt", "job_name", "step_number"],
    na_position="last"
).reset_index(drop=True)

steps_df.to_csv(OUT_DIR / "run_steps_timeline_all.csv", index=False)

# ------------------------------------------------------------
# Filter likely Android / anchor-relevant steps
# ------------------------------------------------------------
anchor_names = {
    "Setup Android Emulator",
    "Run Flutter integration tests",
    "Run Android unit tests",
}

android_mask = (
    steps_df["job_name"].fillna("").str.contains("android", case=False, na=False) |
    steps_df["step_name"].fillna("").isin(anchor_names)
)

android_steps_df = steps_df.loc[android_mask].copy()
android_steps_df.to_csv(OUT_DIR / "run_steps_timeline_android_related.csv", index=False)

# Exact anchor candidates only
anchor_df = steps_df.loc[
    steps_df["step_name"].fillna("").isin(["Setup Android Emulator", "Run Flutter integration tests"])
].copy()
anchor_df.to_csv(OUT_DIR / "run_steps_anchor_candidates.csv", index=False)

# ------------------------------------------------------------
# Earliest start / latest end among anchor candidates
# ------------------------------------------------------------
summary_rows = []

setup_df = anchor_df[anchor_df["step_name"] == "Setup Android Emulator"].copy()
run_df = anchor_df[anchor_df["step_name"] == "Run Flutter integration tests"].copy()

if not setup_df.empty:
    earliest_setup = setup_df.sort_values(["step_started_at_dt", "job_name"]).iloc[0]
    summary_rows.append({
        "type": "earliest_setup_android_emulator",
        "job_name": earliest_setup["job_name"],
        "step_name": earliest_setup["step_name"],
        "step_started_at": earliest_setup["step_started_at"],
        "step_completed_at": earliest_setup["step_completed_at"],
        "job_html_url": earliest_setup["job_html_url"],
    })

if not run_df.empty:
    latest_run = run_df.sort_values(["step_completed_at_dt", "job_name"]).iloc[-1]
    summary_rows.append({
        "type": "latest_run_flutter_integration_tests",
        "job_name": latest_run["job_name"],
        "step_name": latest_run["step_name"],
        "step_started_at": latest_run["step_started_at"],
        "step_completed_at": latest_run["step_completed_at"],
        "job_html_url": latest_run["job_html_url"],
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUT_DIR / "run_anchor_candidate_summary.csv", index=False)

# ------------------------------------------------------------
# Console output
# ------------------------------------------------------------
print("Saved files to:", OUT_DIR.resolve())
print("\nRun metadata:")
print(pd.read_csv(OUT_DIR / "run_metadata.csv").to_string(index=False))

print("\nEarliest/Latest anchor candidates:")
if not summary_df.empty:
    print(summary_df.to_string(index=False))
else:
    print("No anchor candidates found.")

print("\nFirst 20 Android-related steps in actual time order:")
print(android_steps_df.head(20)[[
    "job_name", "step_number", "step_name", "step_started_at", "step_completed_at", "duration_seconds"
]].to_string(index=False))

Saved files to: C:\GitHub\Android-Mobile-Apps\ICST2026_RQ3_Extension\audit_run_20469290792_timeline

Run metadata:
     run_id  run_number  run_attempt    status conclusion        event  workflow_id       run_started_at           updated_at                                 head_sha                                                              html_url
20469290792        1235            1 completed    failure pull_request     43847527 2025-12-23T19:05:25Z 2025-12-23T19:44:45Z a0bb0bf20bed13ff6e40d18f77d539c7c393cdbd https://github.com/bluefireteam/audioplayers/actions/runs/20469290792

Earliest/Latest anchor candidates:
                                type                        job_name                     step_name      step_started_at    step_completed_at                                                                          job_html_url
     earliest_setup_android_emulator call-min-flutter-test / android        Setup Android Emulator 2025-12-23T19:06:18Z 2025-12-23T19:07:30Z https:/

In [1]:
import requests
import pandas as pd
from pathlib import Path

# ============================================================
# CONFIG — set to the manual-review sample you are auditing
# ============================================================
OWNER = "pytorch"
REPO = "executorch"
RUN_ID = 22444580419

# Optional: GitHub token for higher rate limits
GITHUB_TOKEN = None  # e.g. "ghp_xxx"

OUT_DIR = Path(f"audit_run_{RUN_ID}_timeline")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Narrow job-scope hints for the detected Android/community case
# These are used only to find the candidate jobs to inspect.
# ------------------------------------------------------------
JOB_HINTS = [
    "android",
    "emulator",
    "instrument",
    "adb",
    "connectedandroidtest",
    "androidtest",
    "androidjunitrunner",
    "gradle",
    "device",
]

# ------------------------------------------------------------
# Layer-2 phase hints inside the candidate jobs
# ------------------------------------------------------------
PRE_HINTS = [
    "checkout",
    "set up job",
    "setup job",
    "setup",
    "install",
    "dependency",
    "dependencies",
    "prepare",
    "configure",
    "bootstrap",
    "build",
    "compile",
    "gradle",
    "sdk",
    "ndk",
    "java",
    "python",
    "cmake",
    "emulator",
    "start emulator",
    "boot",
    "device",
    "reserve",
    "connect",
]

INVOCATION_HINTS = [
    "instrument",
    "androidjunitrunner",
    "connectedandroidtest",
    "androidtest",
    "integration test",
    "run tests",
    "run test",
    "pytest",
    "unittest",
    "ctest",
    "e2e",
    "benchmark",
    "custom script",
    "script",
    "test",
]

POST_HINTS = [
    "upload",
    "artifact",
    "results",
    "result",
    "report",
    "coverage",
    "collect",
    "cleanup",
    "tear down",
    "teardown",
    "post",
    "complete job",
    "complete",
    "summary",
    "publish",
]

# ============================================================
# API helpers
# ============================================================
def gh_headers():
    headers = {
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
    }
    if GITHUB_TOKEN:
        headers["Authorization"] = f"Bearer {GITHUB_TOKEN}"
    return headers

def gh_get(url, params=None):
    r = requests.get(url, headers=gh_headers(), params=params, timeout=60)
    r.raise_for_status()
    return r.json()

def get_run(owner, repo, run_id):
    url = f"https://api.github.com/repos/{owner}/{repo}/actions/runs/{run_id}"
    return gh_get(url)

def list_jobs(owner, repo, run_id):
    jobs = []
    page = 1
    while True:
        url = f"https://api.github.com/repos/{owner}/{repo}/actions/runs/{run_id}/jobs"
        js = gh_get(url, params={"per_page": 100, "page": page})
        arr = js.get("jobs", [])
        if not arr:
            break
        jobs.extend(arr)
        if len(arr) < 100:
            break
        page += 1
    return jobs

# ============================================================
# Text helpers
# ============================================================
def contains_any(text, phrases):
    t = str(text or "").lower()
    return any(p.lower() in t for p in phrases)

def matched_phrases(text, phrases):
    t = str(text or "").lower()
    return [p for p in phrases if p.lower() in t]

def classify_step_phase(step_name, job_name):
    step_l = str(step_name or "").lower()
    job_l = str(job_name or "").lower()

    post_hits = matched_phrases(step_l, POST_HINTS)
    if post_hits:
        return "post_invocation", "; ".join(post_hits), len(post_hits)

    invocation_hits = matched_phrases(step_l, INVOCATION_HINTS)
    if invocation_hits:
        return "invocation_execution", "; ".join(invocation_hits), len(invocation_hits)

    pre_hits = matched_phrases(step_l, PRE_HINTS)
    if pre_hits:
        return "pre_invocation", "; ".join(pre_hits), len(pre_hits)

    # Fallback for delegated/reusable workflow jobs where the real work
    # may be inside one generic script step.
    if contains_any(job_l, ["android", "instrument", "emulator"]) and contains_any(step_l, ["run", "script"]):
        return "invocation_candidate", "job_context + run/script fallback", 1

    return "other", "", 0

# ============================================================
# Main extraction
# ============================================================
run_js = get_run(OWNER, REPO, RUN_ID)
jobs = list_jobs(OWNER, REPO, RUN_ID)

# ------------------------------------------------------------
# Save run metadata
# ------------------------------------------------------------
pd.DataFrame([{
    "owner": OWNER,
    "repo": REPO,
    "run_id": run_js.get("id"),
    "run_number": run_js.get("run_number"),
    "run_attempt": run_js.get("run_attempt"),
    "status": run_js.get("status"),
    "conclusion": run_js.get("conclusion"),
    "event": run_js.get("event"),
    "workflow_id": run_js.get("workflow_id"),
    "run_started_at": run_js.get("run_started_at"),
    "updated_at": run_js.get("updated_at"),
    "head_sha": run_js.get("head_sha"),
    "html_url": run_js.get("html_url"),
}]).to_csv(OUT_DIR / "run_metadata.csv", index=False)

# ------------------------------------------------------------
# Flatten jobs and steps
# ------------------------------------------------------------
job_rows = []
step_rows = []

for job in jobs:
    job_name = job.get("name")
    labels = ",".join(job.get("labels", [])) if isinstance(job.get("labels"), list) else ""
    steps = job.get("steps", []) or []

    job_rows.append({
        "job_id": job.get("id"),
        "job_name": job_name,
        "job_status": job.get("status"),
        "job_conclusion": job.get("conclusion"),
        "job_started_at": job.get("started_at"),
        "job_completed_at": job.get("completed_at"),
        "job_html_url": job.get("html_url"),
        "runner_name": job.get("runner_name"),
        "labels": labels,
        "n_steps": len(steps),
        "job_hint_match": contains_any(f"{job_name} {labels}", JOB_HINTS),
        "job_hint_terms": "; ".join(matched_phrases(f"{job_name} {labels}", JOB_HINTS)),
    })

    for st in steps:
        step_name = st.get("name")
        phase_guess, phase_reason, phase_score = classify_step_phase(step_name, job_name)

        step_rows.append({
            "job_id": job.get("id"),
            "job_name": job_name,
            "job_status": job.get("status"),
            "job_conclusion": job.get("conclusion"),
            "job_started_at": job.get("started_at"),
            "job_completed_at": job.get("completed_at"),
            "job_html_url": job.get("html_url"),
            "runner_name": job.get("runner_name"),
            "labels": labels,
            "step_number": st.get("number"),
            "step_name": step_name,
            "step_status": st.get("status"),
            "step_conclusion": st.get("conclusion"),
            "step_started_at": st.get("started_at"),
            "step_completed_at": st.get("completed_at"),
            "step_hint_terms": "; ".join(matched_phrases(step_name, JOB_HINTS)),
            "phase_guess": phase_guess,
            "phase_reason": phase_reason,
            "phase_score": phase_score,
        })

jobs_df = pd.DataFrame(job_rows)
steps_df = pd.DataFrame(step_rows)

# ------------------------------------------------------------
# Time fields / durations
# ------------------------------------------------------------
for col in ["job_started_at", "job_completed_at"]:
    jobs_df[col + "_dt"] = pd.to_datetime(jobs_df[col], errors="coerce", utc=True)

jobs_df["job_duration_seconds"] = (
    jobs_df["job_completed_at_dt"] - jobs_df["job_started_at_dt"]
).dt.total_seconds()

for col in ["job_started_at", "job_completed_at", "step_started_at", "step_completed_at"]:
    steps_df[col + "_dt"] = pd.to_datetime(steps_df[col], errors="coerce", utc=True)

steps_df["duration_seconds"] = (
    steps_df["step_completed_at_dt"] - steps_df["step_started_at_dt"]
).dt.total_seconds()

jobs_df = jobs_df.sort_values(["job_started_at_dt", "job_name"], na_position="last").reset_index(drop=True)
steps_df = steps_df.sort_values(
    ["step_started_at_dt", "step_completed_at_dt", "job_name", "step_number"],
    na_position="last",
).reset_index(drop=True)

jobs_df.to_csv(OUT_DIR / "run_jobs.csv", index=False)
steps_df.to_csv(OUT_DIR / "run_steps_timeline_all.csv", index=False)

# ------------------------------------------------------------
# Candidate job scope for the Android/community detection
# Keep only jobs whose name/labels/step names match the Android hints.
# ------------------------------------------------------------
job_scope_ids = set(
    jobs_df.loc[
        jobs_df["job_hint_match"].fillna(False),
        "job_id"
    ].tolist()
)

step_scope_ids = set(
    steps_df.loc[
        steps_df["step_hint_terms"].fillna("").ne(""),
        "job_id"
    ].tolist()
)

candidate_job_ids = sorted(job_scope_ids | step_scope_ids)

candidate_jobs_df = jobs_df[jobs_df["job_id"].isin(candidate_job_ids)].copy()
candidate_steps_df = steps_df[steps_df["job_id"].isin(candidate_job_ids)].copy()

candidate_jobs_df.to_csv(OUT_DIR / "run_jobs_candidate_scope.csv", index=False)
candidate_steps_df.to_csv(OUT_DIR / "run_steps_candidate_scope.csv", index=False)

# ------------------------------------------------------------
# Layer-2 candidate rows inside candidate jobs
# ------------------------------------------------------------
layer2_df = candidate_steps_df[
    candidate_steps_df["phase_guess"].isin(
        ["pre_invocation", "invocation_execution", "post_invocation", "invocation_candidate"]
    )
].copy()

layer2_df.to_csv(OUT_DIR / "run_steps_layer2_candidates.csv", index=False)

# ------------------------------------------------------------
# Per-job Layer-2 summary for manual review
# ------------------------------------------------------------
summary_rows = []

for job_id, g in candidate_steps_df.groupby("job_id", dropna=False):
    g = g.sort_values(["step_number", "step_started_at_dt"], na_position="last").copy()

    pre_df = g[g["phase_guess"] == "pre_invocation"].sort_values(
        ["step_started_at_dt", "step_number"], na_position="last"
    )
    invoke_df = g[g["phase_guess"].isin(["invocation_execution", "invocation_candidate"])].sort_values(
        ["step_started_at_dt", "step_number"], na_position="last"
    )
    post_df = g[g["phase_guess"] == "post_invocation"].sort_values(
        ["step_started_at_dt", "step_number"], na_position="last"
    )

    first_row = g.iloc[0]
    row = {
        "job_id": job_id,
        "job_name": first_row["job_name"],
        "job_html_url": first_row["job_html_url"],
        "runner_name": first_row["runner_name"],
        "job_status": first_row["job_status"],
        "job_conclusion": first_row["job_conclusion"],
        "job_started_at": first_row["job_started_at"],
        "job_completed_at": first_row["job_completed_at"],
        "n_steps_total_in_job": len(g),
        "n_layer2_candidates": len(g[g["phase_guess"] != "other"]),
        "manual_keep_job": "",
        "manual_keep_pre": "",
        "manual_keep_invocation": "",
        "manual_keep_post": "",
        "manual_notes": "",
    }

    if not pre_df.empty:
        x = pre_df.iloc[0]
        row.update({
            "pre_step_number": x["step_number"],
            "pre_step_name": x["step_name"],
            "pre_started_at": x["step_started_at"],
            "pre_completed_at": x["step_completed_at"],
            "pre_reason": x["phase_reason"],
        })
    else:
        row.update({
            "pre_step_number": None,
            "pre_step_name": None,
            "pre_started_at": None,
            "pre_completed_at": None,
            "pre_reason": None,
        })

    if not invoke_df.empty:
        x_first = invoke_df.iloc[0]
        x_last = invoke_df.iloc[-1]
        row.update({
            "invoke_first_step_number": x_first["step_number"],
            "invoke_first_step_name": x_first["step_name"],
            "invoke_first_started_at": x_first["step_started_at"],
            "invoke_first_completed_at": x_first["step_completed_at"],
            "invoke_last_step_number": x_last["step_number"],
            "invoke_last_step_name": x_last["step_name"],
            "invoke_last_started_at": x_last["step_started_at"],
            "invoke_last_completed_at": x_last["step_completed_at"],
            "invoke_reason": x_first["phase_reason"],
        })
    else:
        row.update({
            "invoke_first_step_number": None,
            "invoke_first_step_name": None,
            "invoke_first_started_at": None,
            "invoke_first_completed_at": None,
            "invoke_last_step_number": None,
            "invoke_last_step_name": None,
            "invoke_last_started_at": None,
            "invoke_last_completed_at": None,
            "invoke_reason": None,
        })

    if not post_df.empty:
        x = post_df.iloc[-1]
        row.update({
            "post_step_number": x["step_number"],
            "post_step_name": x["step_name"],
            "post_started_at": x["step_started_at"],
            "post_completed_at": x["step_completed_at"],
            "post_reason": x["phase_reason"],
        })
    else:
        row.update({
            "post_step_number": None,
            "post_step_name": None,
            "post_started_at": None,
            "post_completed_at": None,
            "post_reason": None,
        })

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).sort_values(
    ["job_started_at", "job_name"], na_position="last"
).reset_index(drop=True)

summary_df.to_csv(OUT_DIR / "run_layer2_manual_review_sheet.csv", index=False)

# ------------------------------------------------------------
# Console preview
# ------------------------------------------------------------
print("Saved files to:", OUT_DIR.resolve())

print("\nRun metadata:")
print(pd.read_csv(OUT_DIR / "run_metadata.csv").to_string(index=False))

print("\nCandidate jobs:")
if not candidate_jobs_df.empty:
    print(candidate_jobs_df[[
        "job_name", "job_status", "job_conclusion", "job_started_at",
        "job_completed_at", "job_hint_terms", "job_html_url"
    ]].head(20).to_string(index=False))
else:
    print("No candidate jobs found.")

print("\nLayer-2 candidate steps:")
if not layer2_df.empty:
    print(layer2_df[[
        "job_name", "step_number", "step_name", "phase_guess",
        "phase_reason", "step_started_at", "step_completed_at", "duration_seconds"
    ]].head(40).to_string(index=False))
else:
    print("No Layer-2 candidate steps found.")

print("\nManual-review summary:")
if not summary_df.empty:
    print(summary_df[[
        "job_name",
        "pre_step_name",
        "invoke_first_step_name",
        "post_step_name",
        "manual_keep_job",
        "manual_notes"
    ]].to_string(index=False))
else:
    print("No manual-review summary rows found.")

Saved files to: C:\GitHub\Android-Mobile-Apps\ICST2026_RQ3_Extension\audit_run_22444580419_timeline

Run metadata:
  owner       repo      run_id  run_number  run_attempt    status conclusion event  workflow_id       run_started_at           updated_at                                 head_sha                                                       html_url
pytorch executorch 22444580419       66463            1 completed    failure  push     63454257 2026-02-26T13:39:12Z 2026-02-26T15:13:36Z 5a9b2808411e71a71352005f4568fa3ce0b777a2 https://github.com/pytorch/executorch/actions/runs/22444580419

Candidate jobs:
                                   job_name job_status job_conclusion       job_started_at     job_completed_at    job_hint_terms                                                                   job_html_url
                    android / build-android  completed        success 2026-02-26T13:39:17Z 2026-02-26T13:54:20Z           android https://github.com/pytorch/executorch/actions

In [2]:
import requests
import pandas as pd
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================
OWNER = "bluefireteam"
REPO = "audioplayers"
RUN_ID = 20469493769

# Put your token here if needed for rate limits/private repos
GITHUB_TOKEN = None  # e.g. "ghp_xxx"

OUT_DIR = Path("audit_run_20469493769_timeline")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# API helpers
# ============================================================
def gh_headers():
    headers = {
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
    }
    if GITHUB_TOKEN:
        headers["Authorization"] = f"Bearer {GITHUB_TOKEN}"
    return headers

def gh_get(url, params=None):
    r = requests.get(url, headers=gh_headers(), params=params, timeout=60)
    r.raise_for_status()
    return r.json()

def get_run(owner, repo, run_id):
    url = f"https://api.github.com/repos/{owner}/{repo}/actions/runs/{run_id}"
    return gh_get(url)

def list_jobs(owner, repo, run_id):
    jobs = []
    page = 1
    while True:
        url = f"https://api.github.com/repos/{owner}/{repo}/actions/runs/{run_id}/jobs"
        js = gh_get(url, params={"per_page": 100, "page": page})
        arr = js.get("jobs", [])
        if not arr:
            break
        jobs.extend(arr)
        if len(arr) < 100:
            break
        page += 1
    return jobs

# ============================================================
# Main extraction
# ============================================================
run_js = get_run(OWNER, REPO, RUN_ID)
jobs = list_jobs(OWNER, REPO, RUN_ID)

# Save raw run metadata
pd.DataFrame([{
    "run_id": run_js.get("id"),
    "run_number": run_js.get("run_number"),
    "run_attempt": run_js.get("run_attempt"),
    "status": run_js.get("status"),
    "conclusion": run_js.get("conclusion"),
    "event": run_js.get("event"),
    "workflow_id": run_js.get("workflow_id"),
    "run_started_at": run_js.get("run_started_at"),
    "updated_at": run_js.get("updated_at"),
    "head_sha": run_js.get("head_sha"),
    "html_url": run_js.get("html_url"),
}]).to_csv(OUT_DIR / "run_metadata.csv", index=False)

# ------------------------------------------------------------
# Jobs table
# ------------------------------------------------------------
job_rows = []
step_rows = []

for job in jobs:
    job_rows.append({
        "job_id": job.get("id"),
        "job_name": job.get("name"),
        "job_status": job.get("status"),
        "job_conclusion": job.get("conclusion"),
        "job_started_at": job.get("started_at"),
        "job_completed_at": job.get("completed_at"),
        "job_html_url": job.get("html_url"),
        "runner_name": job.get("runner_name"),
        "labels": ",".join(job.get("labels", [])) if isinstance(job.get("labels"), list) else "",
    })

    for st in job.get("steps", []):
        step_rows.append({
            "job_id": job.get("id"),
            "job_name": job.get("name"),
            "job_started_at": job.get("started_at"),
            "job_completed_at": job.get("completed_at"),
            "job_html_url": job.get("html_url"),
            "step_number": st.get("number"),
            "step_name": st.get("name"),
            "step_status": st.get("status"),
            "step_conclusion": st.get("conclusion"),
            "step_started_at": st.get("started_at"),
            "step_completed_at": st.get("completed_at"),
        })

jobs_df = pd.DataFrame(job_rows)
steps_df = pd.DataFrame(step_rows)

# Save jobs
jobs_df = jobs_df.sort_values(["job_started_at", "job_name"], na_position="last").reset_index(drop=True)
jobs_df.to_csv(OUT_DIR / "run_jobs.csv", index=False)

# Save all steps in actual time order
steps_df["step_started_at_dt"] = pd.to_datetime(steps_df["step_started_at"], errors="coerce", utc=True)
steps_df["step_completed_at_dt"] = pd.to_datetime(steps_df["step_completed_at"], errors="coerce", utc=True)
steps_df["duration_seconds"] = (
    steps_df["step_completed_at_dt"] - steps_df["step_started_at_dt"]
).dt.total_seconds()

steps_df = steps_df.sort_values(
    ["step_started_at_dt", "step_completed_at_dt", "job_name", "step_number"],
    na_position="last"
).reset_index(drop=True)

steps_df.to_csv(OUT_DIR / "run_steps_timeline_all.csv", index=False)

# ------------------------------------------------------------
# Filter likely Android / anchor-relevant steps
# ------------------------------------------------------------
anchor_names = {
    "Setup Android Emulator",
    "Run Flutter integration tests",
    "Run Android unit tests",
}

android_mask = (
    steps_df["job_name"].fillna("").str.contains("android", case=False, na=False) |
    steps_df["step_name"].fillna("").isin(anchor_names)
)

android_steps_df = steps_df.loc[android_mask].copy()
android_steps_df.to_csv(OUT_DIR / "run_steps_timeline_android_related.csv", index=False)

# Exact anchor candidates only
anchor_df = steps_df.loc[
    steps_df["step_name"].fillna("").isin(["Setup Android Emulator", "Run Flutter integration tests"])
].copy()
anchor_df.to_csv(OUT_DIR / "run_steps_anchor_candidates.csv", index=False)

# ------------------------------------------------------------
# Earliest start / latest end among anchor candidates
# ------------------------------------------------------------
summary_rows = []

setup_df = anchor_df[anchor_df["step_name"] == "Setup Android Emulator"].copy()
run_df = anchor_df[anchor_df["step_name"] == "Run Flutter integration tests"].copy()

if not setup_df.empty:
    earliest_setup = setup_df.sort_values(["step_started_at_dt", "job_name"]).iloc[0]
    summary_rows.append({
        "type": "earliest_setup_android_emulator",
        "job_name": earliest_setup["job_name"],
        "step_name": earliest_setup["step_name"],
        "step_started_at": earliest_setup["step_started_at"],
        "step_completed_at": earliest_setup["step_completed_at"],
        "job_html_url": earliest_setup["job_html_url"],
    })

if not run_df.empty:
    latest_run = run_df.sort_values(["step_completed_at_dt", "job_name"]).iloc[-1]
    summary_rows.append({
        "type": "latest_run_flutter_integration_tests",
        "job_name": latest_run["job_name"],
        "step_name": latest_run["step_name"],
        "step_started_at": latest_run["step_started_at"],
        "step_completed_at": latest_run["step_completed_at"],
        "job_html_url": latest_run["job_html_url"],
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUT_DIR / "run_anchor_candidate_summary.csv", index=False)

# ------------------------------------------------------------
# Console output
# ------------------------------------------------------------
print("Saved files to:", OUT_DIR.resolve())
print("\nRun metadata:")
print(pd.read_csv(OUT_DIR / "run_metadata.csv").to_string(index=False))

print("\nEarliest/Latest anchor candidates:")
if not summary_df.empty:
    print(summary_df.to_string(index=False))
else:
    print("No anchor candidates found.")

print("\nFirst 20 Android-related steps in actual time order:")
print(android_steps_df.head(20)[[
    "job_name", "step_number", "step_name", "step_started_at", "step_completed_at", "duration_seconds"
]].to_string(index=False))

Saved files to: C:\GitHub\Android-Mobile-Apps\ICST2026_RQ3_Extension\audit_run_20469493769_timeline

Run metadata:
     run_id  run_number  run_attempt    status conclusion        event  workflow_id       run_started_at           updated_at                                 head_sha                                                              html_url
20469493769        1236            1 completed    failure pull_request     43847527 2025-12-23T19:14:58Z 2025-12-23T19:48:20Z 6c0ce73a84a7e43e5e1a3d91219141ad5fdd4de8 https://github.com/bluefireteam/audioplayers/actions/runs/20469493769

Earliest/Latest anchor candidates:
                                type                        job_name                     step_name      step_started_at    step_completed_at                                                                          job_html_url
     earliest_setup_android_emulator call-min-flutter-test / android        Setup Android Emulator 2025-12-23T19:15:54Z 2025-12-23T19:17:12Z https:/

In [1]:
import requests
import pandas as pd
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================
OWNER = "pytorch"
REPO = "executorch"
RUN_ID = 22444580419

# Put your token here if needed for rate limits/private repos
GITHUB_TOKEN = None  # e.g. "ghp_xxx"

OUT_DIR = Path("audit_run_22444580419_timeline")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# API helpers
# ============================================================
def gh_headers():
    headers = {
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
    }
    if GITHUB_TOKEN:
        headers["Authorization"] = f"Bearer {GITHUB_TOKEN}"
    return headers

def gh_get(url, params=None):
    r = requests.get(url, headers=gh_headers(), params=params, timeout=60)
    r.raise_for_status()
    return r.json()

def get_run(owner, repo, run_id):
    url = f"https://api.github.com/repos/{owner}/{repo}/actions/runs/{run_id}"
    return gh_get(url)

def list_jobs(owner, repo, run_id):
    jobs = []
    page = 1
    while True:
        url = f"https://api.github.com/repos/{owner}/{repo}/actions/runs/{run_id}/jobs"
        js = gh_get(url, params={"per_page": 100, "page": page})
        arr = js.get("jobs", [])
        if not arr:
            break
        jobs.extend(arr)
        if len(arr) < 100:
            break
        page += 1
    return jobs

# ============================================================
# Main extraction
# ============================================================
run_js = get_run(OWNER, REPO, RUN_ID)
jobs = list_jobs(OWNER, REPO, RUN_ID)

# Save raw run metadata
pd.DataFrame([{
    "run_id": run_js.get("id"),
    "run_number": run_js.get("run_number"),
    "run_attempt": run_js.get("run_attempt"),
    "status": run_js.get("status"),
    "conclusion": run_js.get("conclusion"),
    "event": run_js.get("event"),
    "workflow_id": run_js.get("workflow_id"),
    "run_started_at": run_js.get("run_started_at"),
    "updated_at": run_js.get("updated_at"),
    "head_sha": run_js.get("head_sha"),
    "html_url": run_js.get("html_url"),
}]).to_csv(OUT_DIR / "run_metadata.csv", index=False)

# ------------------------------------------------------------
# Jobs table
# ------------------------------------------------------------
job_rows = []
step_rows = []

for job in jobs:
    job_rows.append({
        "job_id": job.get("id"),
        "job_name": job.get("name"),
        "job_status": job.get("status"),
        "job_conclusion": job.get("conclusion"),
        "job_started_at": job.get("started_at"),
        "job_completed_at": job.get("completed_at"),
        "job_html_url": job.get("html_url"),
        "runner_name": job.get("runner_name"),
        "labels": ",".join(job.get("labels", [])) if isinstance(job.get("labels"), list) else "",
    })

    for st in job.get("steps", []):
        step_rows.append({
            "job_id": job.get("id"),
            "job_name": job.get("name"),
            "job_started_at": job.get("started_at"),
            "job_completed_at": job.get("completed_at"),
            "job_html_url": job.get("html_url"),
            "step_number": st.get("number"),
            "step_name": st.get("name"),
            "step_status": st.get("status"),
            "step_conclusion": st.get("conclusion"),
            "step_started_at": st.get("started_at"),
            "step_completed_at": st.get("completed_at"),
        })

jobs_df = pd.DataFrame(job_rows)
steps_df = pd.DataFrame(step_rows)

# Save jobs
jobs_df = jobs_df.sort_values(["job_started_at", "job_name"], na_position="last").reset_index(drop=True)
jobs_df.to_csv(OUT_DIR / "run_jobs.csv", index=False)

# Save all steps in actual time order
steps_df["step_started_at_dt"] = pd.to_datetime(steps_df["step_started_at"], errors="coerce", utc=True)
steps_df["step_completed_at_dt"] = pd.to_datetime(steps_df["step_completed_at"], errors="coerce", utc=True)
steps_df["duration_seconds"] = (
    steps_df["step_completed_at_dt"] - steps_df["step_started_at_dt"]
).dt.total_seconds()

steps_df = steps_df.sort_values(
    ["step_started_at_dt", "step_completed_at_dt", "job_name", "step_number"],
    na_position="last"
).reset_index(drop=True)

steps_df.to_csv(OUT_DIR / "run_steps_timeline_all.csv", index=False)

# ------------------------------------------------------------
# Filter likely Android / anchor-relevant steps
# ------------------------------------------------------------
anchor_names = {
    "Run Android emulator",
    "Post AVD cache",
    "AVD cache",
    "Run tests",
    "Run custom android tests",
}

android_mask = (
    steps_df["job_name"].fillna("").str.contains("android|emulator", case=False, na=False) |
    steps_df["step_name"].fillna("").isin(anchor_names) |
    steps_df["step_name"].fillna("").str.contains("android|emulator|avd|test", case=False, na=False)
)

android_steps_df = steps_df.loc[android_mask].copy()
android_steps_df.to_csv(OUT_DIR / "run_steps_timeline_android_related.csv", index=False)

# Exact anchor candidates only
anchor_df = steps_df.loc[
    steps_df["step_name"].fillna("").isin([
        "Run Android emulator",
        "Post AVD cache",
        "AVD cache",
        "Run tests",
        "Run custom android tests",
    ])
].copy()
anchor_df.to_csv(OUT_DIR / "run_steps_anchor_candidates.csv", index=False)

# ------------------------------------------------------------
# Earliest start / latest end among anchor candidates
# ------------------------------------------------------------
summary_rows = []

start_df = anchor_df[anchor_df["step_name"] == "Run Android emulator"].copy()
end_df = anchor_df[anchor_df["step_name"].isin(["Post AVD cache", "Run tests", "Run custom android tests"])].copy()

if not start_df.empty:
    earliest_start = start_df.sort_values(["step_started_at_dt", "job_name"]).iloc[0]
    summary_rows.append({
        "type": "earliest_run_android_emulator",
        "job_name": earliest_start["job_name"],
        "step_name": earliest_start["step_name"],
        "step_started_at": earliest_start["step_started_at"],
        "step_completed_at": earliest_start["step_completed_at"],
        "job_html_url": earliest_start["job_html_url"],
    })

if not end_df.empty:
    latest_end = end_df.sort_values(["step_completed_at_dt", "job_name"]).iloc[-1]
    summary_rows.append({
        "type": "latest_candidate_execution_end",
        "job_name": latest_end["job_name"],
        "step_name": latest_end["step_name"],
        "step_started_at": latest_end["step_started_at"],
        "step_completed_at": latest_end["step_completed_at"],
        "job_html_url": latest_end["job_html_url"],
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUT_DIR / "run_anchor_candidate_summary.csv", index=False)

# ------------------------------------------------------------
# Console output
# ------------------------------------------------------------
print("Saved files to:", OUT_DIR.resolve())
print("\nRun metadata:")
print(pd.read_csv(OUT_DIR / "run_metadata.csv").to_string(index=False))

print("\nEarliest/Latest anchor candidates:")
if not summary_df.empty:
    print(summary_df.to_string(index=False))
else:
    print("No anchor candidates found.")

print("\nFirst 20 Android-related steps in actual time order:")
print(android_steps_df.head(20)[[
    "job_name", "step_number", "step_name", "step_started_at", "step_completed_at", "duration_seconds"
]].to_string(index=False))

Saved files to: C:\GitHub\Android-Mobile-Apps\ICST2026_RQ3_Extension\audit_run_22444580419_timeline

Run metadata:
     run_id  run_number  run_attempt    status conclusion event  workflow_id       run_started_at           updated_at                                 head_sha                                                       html_url
22444580419       66463            1 completed    failure  push     63454257 2026-02-26T13:39:12Z 2026-02-26T15:13:36Z 5a9b2808411e71a71352005f4568fa3ce0b777a2 https://github.com/pytorch/executorch/actions/runs/22444580419

Earliest/Latest anchor candidates:
                          type               job_name            step_name      step_started_at    step_completed_at                                                                   job_html_url
 earliest_run_android_emulator android / run-emulator Run Android emulator 2026-02-26T13:54:44Z 2026-02-26T14:38:55Z https://github.com/pytorch/executorch/actions/runs/22444580419/job/64997440976
latest_cand